# Control de un player de vídeo a través de gestos

## Paquetes necesarios

In [4]:
import cv2
import mediapipe as mp

![Hand landmarks.]("C:/Users/34611/OneDrive/Escritorio/GTDM/Tercero/Imagen/Proyecto/Control-de-player-via-gestos/images/hand_landmarks.png")

In [5]:
def get_finger_states(hand_landmarks):
    fingers = []

    # Pulgar (compara X en lugar de Y porque el pulgar se mueve lateralmente)
    if hand_landmarks.landmark[4].x > hand_landmarks.landmark[3].x:
        fingers.append(1)  # Extendido
    else:
        fingers.append(0)  # Doblado

    # Resto de los dedos: Índice, Medio, Anular, Meñique
    tips = [8, 12, 16, 20]
    pips = [6, 10, 14, 18]

    for tip, pip in zip(tips, pips):
        if hand_landmarks.landmark[tip].y < hand_landmarks.landmark[pip].y:
            fingers.append(1)
        else:
            fingers.append(0)

    return fingers  # [pulgar, índice, medio, anular, meñique]

In [6]:
# Detecta gesto: Mano abierta o puño
def detect_custom_gesture(fingers):
    # fingers = [pulgar, índice, medio, anular, meñique]
    #if fingers == [1, 1, 0, 0, 0]:
     #   return "L"
    #elif fingers == [0, 1, 1, 0, 0]:
      #  return "V"
    if fingers == [0, 1, 0, 0, 1]:
        return "Rock"
    #elif fingers == [0, 0, 0, 0, 0]:
     #   return "Puño"
    if fingers == [1, 1, 1, 1, 1]:
        return "Palma"
    else:
        return "Unknown"

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

# Configurar Selenium (modo visible o headless)
options = Options()
# options.add_argument('--headless')  # descomenta si quieres que sea invisible
options.add_argument("--autoplay-policy=no-user-gesture-required")

#WebDriverWait(driver, 10).until(
#    lambda d: d.execute_script("return video.readyState;") == 4
#)

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=136.0.7103.114)
Stacktrace:
	GetHandleVerifier [0x00007FF6EE18CF45+75717]
	GetHandleVerifier [0x00007FF6EE18CFA0+75808]
	(No symbol) [0x00007FF6EDF58F9A]
	(No symbol) [0x00007FF6EDF31B41]
	(No symbol) [0x00007FF6EDFDEC2E]
	(No symbol) [0x00007FF6EDFFECC2]
	(No symbol) [0x00007FF6EDFD7153]
	(No symbol) [0x00007FF6EDFA0421]
	(No symbol) [0x00007FF6EDFA11B3]
	GetHandleVerifier [0x00007FF6EE48D71D+3223453]
	GetHandleVerifier [0x00007FF6EE487CC2+3200322]
	GetHandleVerifier [0x00007FF6EE4A5AF3+3322739]
	GetHandleVerifier [0x00007FF6EE1A6A1A+180890]
	GetHandleVerifier [0x00007FF6EE1AE11F+211359]
	GetHandleVerifier [0x00007FF6EE195294+109332]
	GetHandleVerifier [0x00007FF6EE195442+109762]
	GetHandleVerifier [0x00007FF6EE17BA59+4825]
	BaseThreadInitThunk [0x00007FFC10547374+20]
	RtlUserThreadStart [0x00007FFC11A3CC91+33]


In [2]:
import pyautogui

In [13]:
driver = webdriver.Chrome(options=options)
driver.get("https://areyousure.wavedash.com")


In [55]:
def control(modo,pos_anterior,punta_indice,mov):
    global driver
    global previousMilis
    if modo == 'navegación':
        screen_w, screen_h = pyautogui.size()  #Dimensiones de la pantalla en píxeles
        if pos_anterior != None:
            if punta_indice.y-pos_anterior[8].y > 0.1:
                pyautogui.click()
                from selenium.webdriver.support.ui import WebDriverWait
                actual_url = driver.current_url 
                print(actual_url)
                if actual_url != 'areyousure.wavedash.com':
                    return 'player'
            else:  #Para que no haga el click y el movimiento a la vez
                pos_X = int(punta_indice.x*screen_w)
                pos_Y = int(punta_indice.y*screen_h)
                pyautogui.moveTo(pos_X, pos_Y)
        return 'navegación'
    else:
        currentMilis = time.time()
        if pos_anterior != None and currentMilis-previousMilis>0.5:
            if punta_indice.x-pos_anterior[8].x > 0.04:
                print('adelantar')
                mov = 'der-izq'
                driver.execute_script("video.currentTime += 5;")
            elif punta_indice.x-pos_anterior[8].x < -0.04:
                print('atrasar')
                mov = 'izq-der'
                driver.execute_script("video.currentTime -= 5;")
            if punta_indice.y-pos_anterior[8].y < -0.1:
                print('subir volumen')
                mov = 'abajo-arriba'
                driver.execute_script("if (video.volume <= 0.9){video.volume += 0.1}else {video.volume=1};")
            elif punta_indice.y-pos_anterior[8].y > 0.1:
                print('bajar volumen')
                mov = 'arriba-abajo'
                driver.execute_script("if (video.volume >= 0.1){video.volume -= 0.1}else {video.volume=0};")
            if mov == 'Palma':
                print('parar')
                driver.execute_script("video.pause();")
            elif mov == 'Rock':
                print('continuar')
                driver.execute_script("video.play();")
                
            if mov != 'Unknown':
                print('indice y:',punta_indice.y)
                print('movimiento punta_indice x', punta_indice.x-pos_anterior[8].x ,punta_indice.y-pos_anterior[8].y)
                previousMilis = currentMilis
        cv2.putText(image, f'Gesture: {mov}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        return 'player'

In [56]:
import time
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
cap = cv2.VideoCapture(1)  #CAMBIAR A 0 SI NO VA
pos_anterior = None
modo = 'navegación'
currentMilis = 0
previousMilis = 0
with mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print('No se puede leer la cámara')
            break

        try:
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = hands.process(image)
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    punta_indice = hand_landmarks.landmark[8]
                    dedos = get_finger_states(hand_landmarks)
                    mov = detect_custom_gesture(dedos) # Detecta el gesto de la palma 
                    modo = control(modo,pos_anterior,punta_indice,mov)
                    pos_anterior = hand_landmarks.landmark
                    
            cv2.imshow('Hand Gesture Recognition', image)
        except Exception as e:
            import traceback
            traceback.print_exc()
            
        
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


https://areyousure.wavedash.com/player.html?content=Video2/master.m3u8
adelantar
subir volumen
indice y: 0.26160505414009094
movimiento punta_indice x 0.041307538747787476 -0.29131147265434265
parar
indice y: 0.24474970996379852
movimiento punta_indice x 0.002490699291229248 0.003705129027366638
parar
indice y: 0.25104379653930664
movimiento punta_indice x 0.003415524959564209 0.00279843807220459
parar
indice y: 0.2686472237110138
movimiento punta_indice x 8.207559585571289e-05 -0.00032576918601989746
adelantar
indice y: 0.2447175830602646
movimiento punta_indice x 0.04702126979827881 0.0190725177526474
parar
indice y: 0.2890161871910095
movimiento punta_indice x 0.0003236532211303711 -0.0002957582473754883
parar
indice y: 0.28492671251296997
movimiento punta_indice x -0.004786491394042969 -0.0029866695404052734
parar
indice y: 0.2633823752403259
movimiento punta_indice x 0.02818167209625244 0.007948338985443115
parar
indice y: 0.32576942443847656
movimiento punta_indice x 0.0112270712